# MS-Dialog Data Preparation

Extracts 100 high-quality tech-support dialogs from **MSDialog-Intent** for the clarifying-question uncertainty experiment.

## Experiment mapping (parallel to MedQA)

| MedQA field | MS-Dialog equivalent |
|---|---|
| `ehr_summary` | `original_question` (user's OQ utterance) |
| `question` | fixed prompt: *"What is the most helpful solution to this user's problem?"* |
| `options` | N/A — open-ended, not MCQ |
| `simulator_context` | User FD utterances + Agent CQ utterances (hidden context) |
| `correct_answer` | `accepted_answer` (utterance(s) with `is_answer=1`) |

## Tag legend
| Tag | Meaning |
|---|---|
| **OQ** | Original Question — the user's problem statement |
| **PA** | Potential Answer — agent's proposed solution |
| **FD** | Further Details — user elaborates on the problem |
| **CQ** | Clarifying Question — agent (or user) asks for more info |
| **IR** | Information Request — similar to CQ |
| **GG** | Gratitude / Greeting |
| **PF** | Positive Feedback |
| **NF** | Negative Feedback |

## Output
`datasets/ms-dialog/msdialog_100.jsonl` — one JSON object per line with fields:
```
case_id, title, category, original_question,
simulator_context, accepted_answer, n_utterances, has_fd, has_agent_cq
```

In [1]:
import sys
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = Path("../../").resolve()
DATA_DIR   = ROOT / "datasets" / "ms-dialog"
OUT_DIR    = DATA_DIR

INTENT_JSON = DATA_DIR / "MSDialog-Intent.json"
OUTPUT_JSONL = OUT_DIR / "msdialog_100.jsonl"

N_SAMPLES   = 100   # total examples to extract
RANDOM_SEED = 42

sys.path.insert(0, str(ROOT))
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT        : {ROOT}")
print(f"Input JSON  : {INTENT_JSON}")
print(f"Output JSONL: {OUTPUT_JSONL}")

ROOT        : D:\final_project\pilot_study
Input JSON  : D:\final_project\pilot_study\datasets\ms-dialog\MSDialog-Intent.json
Output JSONL: D:\final_project\pilot_study\datasets\ms-dialog\msdialog_100.jsonl


## 1. Load Dataset

In [2]:
import json
from collections import Counter, defaultdict

with open(INTENT_JSON, encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Total dialogs loaded: {len(raw_data)}")

# ── Inspect tag distribution ──────────────────────────────────────────────────
tag_counts = Counter()
for dialog in raw_data.values():
    for u in dialog.get("utterances", []):
        tags = u.get("tags", "").split() if isinstance(u.get("tags", ""), str) else u.get("tags", [])
        tag_counts.update(tags)

print("\nTag distribution across all utterances:")
for tag, count in tag_counts.most_common():
    print(f"  {tag:5s}: {count:5d}")

Total dialogs loaded: 2199

Tag distribution across all utterances:
  GG   :  4004
  PA   :  3980
  FD   :  2481
  OQ   :  2339
  PF   :  1076
  IR   :  1076
  FQ   :   877
  NF   :   756
  CQ   :   749
  RQ   :   606
  JK   :   256
  O    :   149


## 2. Define Extraction Helpers

**`get_tags(u)`** — normalises both string and list tag formats.

**`extract_fields(dialog_id, dialog)`** — returns a structured dict or `None` if the dialog fails quality filters.

### Quality filters
1. Must have at least one **OQ** utterance with ≥ 15 words  
2. Must have at least one **accepted answer** (`is_answer == 1`)  
3. Must have ≥ 3 utterances (enough context for a simulator to respond to a follow-up CQ)

### Simulator context construction
The simulator context gives the model-as-simulator enough information to answer any clarifying question the clinician/model generates. It is built from:
- All **User FD** utterances — the user's own further details  
- All **Agent CQ** utterances — questions the support agent originally asked  

These two sources together let the simulator roleplay as the user with full knowledge of the problem details and what information was sought. They are separated by `---` so the simulator knows the source.

### Accepted-answer fallback chain
| Priority | Rule |
|---|---|
| 1 | First PA utterance with `is_answer == 1` (explicit mark) |
| 2 | All PA utterances with `is_answer == 1` joined if multiple |
| 3 | Last PA utterance from Agent (no explicit mark — use heuristic) |

In [3]:
def get_tags(utterance: dict) -> list[str]:
    """Return list of tags, handling both str and list formats."""
    raw = utterance.get("tags", "")
    if isinstance(raw, list):
        return raw
    return raw.split() if raw else []


def extract_fields(dialog_id: str, dialog: dict) -> dict | None:
    """Parse one dialog into the experiment record schema.
    
    Returns None if the dialog does not meet quality thresholds.
    """
    utts = dialog.get("utterances", [])

    if len(utts) < 3:
        return None

    # ── Buckets by role & tag ─────────────────────────────────────────────────
    oq_texts        = []   # User OQ utterances
    user_fd_texts   = []   # User FD — further details the user provided
    agent_cq_texts  = []   # Agent CQ — clarifying questions the agent asked
    answer_texts    = []   # PA utterances with is_answer == 1
    pa_fallback     = []   # All Agent PA utterances (for fallback)

    for u in utts:
        tags   = get_tags(u)
        text   = u.get("utterance", "").strip()
        actor  = u.get("actor_type", "")
        marked = u.get("is_answer", 0) == 1

        if not text:
            continue

        if "OQ" in tags and actor == "User":
            oq_texts.append(text)

        if "FD" in tags and actor == "User":
            user_fd_texts.append(text)

        if "CQ" in tags and actor == "Agent":
            agent_cq_texts.append(text)

        if "PA" in tags and actor == "Agent":
            pa_fallback.append(text)
            if marked:
                answer_texts.append(text)

    # ── Quality gates ─────────────────────────────────────────────────────────
    if not oq_texts:
        return None  # no OQ utterance

    original_question = " ".join(oq_texts)  # merge if multiple OQ utterances
    if len(original_question.split()) < 15:
        return None  # too short to be informative

    # ── Accepted answer (with fallback) ───────────────────────────────────────
    if answer_texts:
        accepted_answer = "\n\n".join(answer_texts)  # may be multiple marked PAs
        answer_source   = "explicit_is_answer"
    elif pa_fallback:
        accepted_answer = pa_fallback[-1]   # last PA — typically the resolution
        answer_source   = "fallback_last_pa"
    else:
        return None  # no usable answer at all

    # ── Simulator context ─────────────────────────────────────────────────────
    sim_parts = []
    if user_fd_texts:
        sim_parts.append(
            "[Further details provided by the user]\n"
            + "\n\n".join(user_fd_texts)
        )
    if agent_cq_texts:
        sim_parts.append(
            "[Clarifying questions asked by the support agent]\n"
            + "\n\n".join(agent_cq_texts)
        )
    simulator_context = "\n\n---\n\n".join(sim_parts) if sim_parts else ""

    return {
        "case_id":            dialog_id,
        "title":              dialog.get("title", ""),
        "category":           dialog.get("category", ""),
        "original_question":  original_question,
        "simulator_context":  simulator_context,
        "accepted_answer":    accepted_answer,
        "answer_source":      answer_source,
        "n_utterances":       len(utts),
        "has_fd":             bool(user_fd_texts),
        "has_agent_cq":       bool(agent_cq_texts),
    }


print("Helper functions defined.")

Helper functions defined.


## 3. Filter Quality Dialogs

In [4]:
all_records = []
skip_reasons = Counter()

for dialog_id, dialog in raw_data.items():
    rec = extract_fields(dialog_id, dialog)
    if rec is None:
        skip_reasons["failed_quality_filter"] += 1
    else:
        all_records.append(rec)

print(f"Dialogs passing quality filter : {len(all_records)} / {len(raw_data)}")
print(f"Skipped                        : {skip_reasons['failed_quality_filter']}")

# ── Context richness breakdown ────────────────────────────────────────────────
has_fd_count      = sum(r["has_fd"]        for r in all_records)
has_cq_count      = sum(r["has_agent_cq"]  for r in all_records)
has_both          = sum(r["has_fd"] and r["has_agent_cq"] for r in all_records)
explicit_answers  = sum(r["answer_source"] == "explicit_is_answer" for r in all_records)
fallback_answers  = sum(r["answer_source"] == "fallback_last_pa"   for r in all_records)

print(f"\nSimulator context richness:")
print(f"  Has user FD                  : {has_fd_count} ({has_fd_count/len(all_records):.1%})")
print(f"  Has agent CQ                 : {has_cq_count} ({has_cq_count/len(all_records):.1%})")
print(f"  Has both FD + CQ             : {has_both} ({has_both/len(all_records):.1%})")
print(f"\nAnswer source:")
print(f"  Explicit is_answer==1        : {explicit_answers} ({explicit_answers/len(all_records):.1%})")
print(f"  Fallback (last PA)           : {fallback_answers} ({fallback_answers/len(all_records):.1%})")

# ── Category distribution ─────────────────────────────────────────────────────
cat_counts = Counter(r["category"] for r in all_records)
print(f"\nTop 15 categories:")
for cat, cnt in cat_counts.most_common(15):
    print(f"  {cat:35s}: {cnt}")

Dialogs passing quality filter : 1995 / 2199
Skipped                        : 204

Simulator context richness:
  Has user FD                  : 920 (46.1%)
  Has agent CQ                 : 345 (17.3%)
  Has both FD + CQ             : 180 (9.0%)

Answer source:
  Explicit is_answer==1        : 1809 (90.7%)
  Fallback (last PA)           : 186 (9.3%)

Top 15 categories:
  PowerPoint                         : 190
  Excel                              : 177
  Bing_Apps                          : 152
  Bing_Search                        : 151
  Windows_7                          : 138
  Word                               : 131
  Bing_Maps                          : 131
  Skype_iOS                          : 123
  Skype_Android                      : 113
  Skype_Mac                          : 92
  Windows_RT_8.1                     : 92
  Windows_8.1                        : 91
  Skype_Windows_Desktop              : 86
  Apps_Windows_10                    : 79
  Skype_Windows_10              

## 4. Stratified Sampling — 100 Examples

We prioritise dialogs with **both FD and agent CQ** (richest simulator context) then fill remaining slots proportionally from categories to preserve domain variety.

Sampling priority:
1. Dialogs with FD + CQ (richest context) — proportional by category  
2. Dialogs with at least FD or CQ — proportional by category  
3. Remaining (OQ + answer only) — proportional by category

In [5]:
import random
random.seed(RANDOM_SEED)

# ── Tier assignment ───────────────────────────────────────────────────────────
tier1 = [r for r in all_records if r["has_fd"] and r["has_agent_cq"]]  # richest
tier2 = [r for r in all_records if (r["has_fd"] or r["has_agent_cq"]) and not (r["has_fd"] and r["has_agent_cq"])]
tier3 = [r for r in all_records if not r["has_fd"] and not r["has_agent_cq"]]

print(f"Tier 1 (FD + CQ)  : {len(tier1)}")
print(f"Tier 2 (FD or CQ) : {len(tier2)}")
print(f"Tier 3 (OQ only)  : {len(tier3)}")


def stratified_sample(pool: list, n: int, category_key: str = "category", seed: int = 42) -> list:
    """Proportional stratified sample from pool, capped at n."""
    if not pool or n <= 0:
        return []
    n = min(n, len(pool))

    by_cat = defaultdict(list)
    for r in pool:
        by_cat[r[category_key]].append(r)

    # Proportional allocation
    total = len(pool)
    allocations = {cat: max(1, round(n * len(items) / total)) for cat, items in by_cat.items()}

    # Adjust to exactly n
    diff = sum(allocations.values()) - n
    cats_sorted = sorted(allocations, key=lambda c: -len(by_cat[c]))
    for i, cat in enumerate(cats_sorted):
        if diff == 0:
            break
        delta = 1 if diff > 0 else -1
        allocations[cat] = max(0, allocations[cat] - delta)
        diff -= delta if diff > 0 else -delta

    sampled = []
    rng = random.Random(seed)
    for cat, items in by_cat.items():
        k = min(allocations[cat], len(items))
        sampled.extend(rng.sample(items, k))

    return sampled


# ── Sample: fill 100 from tier1 first, supplement from tier2, then tier3 ─────
n_target = N_SAMPLES

sample1 = stratified_sample(tier1, n=min(n_target, len(tier1)))
remaining = n_target - len(sample1)

sample2 = stratified_sample(tier2, n=min(remaining, len(tier2))) if remaining > 0 else []
remaining -= len(sample2)

sample3 = stratified_sample(tier3, n=min(remaining, len(tier3))) if remaining > 0 else []

selected = sample1 + sample2 + sample3
random.shuffle(selected)  # randomise order before writing

print(f"\nSample composition:")
print(f"  From tier 1 (FD + CQ) : {len(sample1)}")
print(f"  From tier 2 (FD or CQ): {len(sample2)}")
print(f"  From tier 3 (OQ only) : {len(sample3)}")
print(f"  Total                 : {len(selected)}")

# ── Check no duplicates ───────────────────────────────────────────────────────
ids = [r["case_id"] for r in selected]
assert len(ids) == len(set(ids)), "Duplicate case_ids in sample!"
print("No duplicate case_ids — check PASSED.")

Tier 1 (FD + CQ)  : 180
Tier 2 (FD or CQ) : 905
Tier 3 (OQ only)  : 910

Sample composition:
  From tier 1 (FD + CQ) : 100
  From tier 2 (FD or CQ): 0
  From tier 3 (OQ only) : 0
  Total                 : 100
No duplicate case_ids — check PASSED.


## 5. Dataset Inspection — Sample Examples

In [6]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(selected)

print(f"Dataset shape: {df.shape}")
print(f"\nCategory distribution:")
display(df["category"].value_counts().head(20))

print(f"\nAnswer source breakdown:")
display(df["answer_source"].value_counts())

print(f"\nSimulator context richness:")
print(f"  Has FD context  : {df['has_fd'].sum()} ({df['has_fd'].mean():.1%})")
print(f"  Has agent CQ    : {df['has_agent_cq'].sum()} ({df['has_agent_cq'].mean():.1%})")

print(f"\nText length stats:")
df["oq_words"]   = df["original_question"].apply(lambda x: len(x.split()))
df["ctx_words"]  = df["simulator_context"].apply(lambda x: len(x.split()) if x else 0)
df["ans_words"]  = df["accepted_answer"].apply(lambda x: len(x.split()))
display(df[["oq_words", "ctx_words", "ans_words"]].describe().round(1))

Dataset shape: (100, 10)

Category distribution:


category
PowerPoint               15
Bing_Apps                12
Excel                    11
Windows_7                 7
Word                      7
Bing_Maps                 7
Bing_Search               6
Apps_Windows_10           5
Bing                      4
Windows_8.1               4
Skype_iOS                 4
Windows_10                3
Skype_Android             3
Windows_RT_8.1            3
Skype_Windows_Desktop     2
Skype_Mac                 2
Skype_Windows_10          2
Outlook_Email             2
Games_Windows_10          1
Name: count, dtype: int64


Answer source breakdown:


answer_source
explicit_is_answer    86
fallback_last_pa      14
Name: count, dtype: int64


Simulator context richness:
  Has FD context  : 100 (100.0%)
  Has agent CQ    : 100 (100.0%)

Text length stats:


,oq_words,ctx_words,ans_words
count,100.0,100.0,100.0
mean,84.2,197.1,81.0
std,60.6,138.7,58.8
min,19.0,21.0,9.0
25%,40.5,105.0,42.8
50%,68.0,161.5,64.5
75%,106.2,242.5,111.5
max,299.0,691.0,354.0


In [7]:
# ── Spot-check 3 examples ────────────────────────────────────────────────────
for rec in selected[:3]:
    print("=" * 70)
    print(f"case_id  : {rec['case_id']}")
    print(f"title    : {rec['title']}")
    print(f"category : {rec['category']}")
    print(f"answer_source: {rec['answer_source']}")
    print()
    print("ORIGINAL QUESTION:")
    print(rec["original_question"][:400])
    print()
    if rec["simulator_context"]:
        print("SIMULATOR CONTEXT (first 400 chars):")
        print(rec["simulator_context"][:400])
    else:
        print("SIMULATOR CONTEXT: (empty — OQ-only dialog)")
    print()
    print("ACCEPTED ANSWER (first 300 chars):")
    print(rec["accepted_answer"][:300])
    print()

case_id  : 3664
title    : Redirect from Bing Search
category : Bing_Search
answer_source: fallback_last_pa

ORIGINAL QUESTION:
Hello, Today, I was searching the web and landed on a website, not the one I intended to visit. I do believe this must be some sort of bug with a search function. The website is http://kitagawa.com, which is a legitimate website. When you type Kitagawa into a Bing search, the first result is the official website, as shown below. When you scroll over the page, the web browser status states the corr

SIMULATOR CONTEXT (first 400 chars):
[Further details provided by the user]
Hello,  After completing more research about the issue, it seemed as if a probable cause was a hack into the .htaccess file or sitemap.xml, with nothing to do with the search engine at all. I am presuming that the issue was fixed, as there were other cases by other people of this happening also. And I had already removed all browsing history, cookies, caches,

ACCEPTED ANSWER (first 300 char

## 6. Simulator Context Completeness Check

For dialogs where `simulator_context` is empty (OQ-only tier-3 cases), the simulator has no hidden context to draw from when answering a clarifying question. We flag these — they are still usable (the simulator must improvise based on OQ alone) but they're lower quality.

We also compute a richness score for transparency.

In [8]:
empty_ctx = [r for r in selected if not r["simulator_context"].strip()]
print(f"Dialogs with empty simulator context: {len(empty_ctx)} / {len(selected)}")

if empty_ctx:
    print("\nThese cases have OQ + accepted answer but no FD or agent CQ.")
    print("The simulator will respond based on OQ context alone.")
    for r in empty_ctx[:3]:
        print(f"  [{r['case_id']}] {r['title'][:60]}")

# ── Richness score (0–2) for each record ─────────────────────────────────────
for r in selected:
    r["context_richness"] = int(r["has_fd"]) + int(r["has_agent_cq"])

from collections import Counter
richness_dist = Counter(r["context_richness"] for r in selected)
print(f"\nContext richness distribution (0=none, 1=FD or CQ, 2=both):")
for score in sorted(richness_dist):
    print(f"  Score {score}: {richness_dist[score]} records")

Dialogs with empty simulator context: 0 / 100

Context richness distribution (0=none, 1=FD or CQ, 2=both):
  Score 2: 100 records


## 7. Save Output

Write `msdialog_100.jsonl` — one JSON object per line, ordered fields matching MedQA schema where applicable.

In [9]:
# ── Assign sequential case_ids for cleaner downstream referencing ─────────────
output_records = []
for i, rec in enumerate(selected, start=1):
    output_records.append({
        "case_id":            f"msd_{i:03d}",
        "source_dialog_id":   rec["case_id"],      # original MSDialog ID
        "title":              rec["title"],
        "category":           rec["category"],
        "original_question":  rec["original_question"],
        "simulator_context":  rec["simulator_context"],
        "accepted_answer":    rec["accepted_answer"],
        "answer_source":      rec["answer_source"],
        "context_richness":   rec["context_richness"],
        "n_utterances":       rec["n_utterances"],
        "has_fd":             rec["has_fd"],
        "has_agent_cq":       rec["has_agent_cq"],
    })

with open(OUTPUT_JSONL, "w", encoding="utf-8") as f:
    for rec in output_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(output_records)} records to {OUTPUT_JSONL}")

# ── Verify round-trip ─────────────────────────────────────────────────────────
with open(OUTPUT_JSONL, encoding="utf-8") as f:
    loaded = [json.loads(l) for l in f if l.strip()]

assert len(loaded) == N_SAMPLES, f"Expected {N_SAMPLES}, got {len(loaded)}"
print(f"Round-trip verification PASSED — {len(loaded)} records.")
print(f"\nFirst record keys: {list(loaded[0].keys())}")

Saved 100 records to D:\final_project\pilot_study\datasets\ms-dialog\msdialog_100.jsonl
Round-trip verification PASSED — 100 records.

First record keys: ['case_id', 'source_dialog_id', 'title', 'category', 'original_question', 'simulator_context', 'accepted_answer', 'answer_source', 'context_richness', 'n_utterances', 'has_fd', 'has_agent_cq']


## 8. Final Summary

In [10]:
out_df = pd.DataFrame(output_records)

print("=" * 60)
print("MS-DIALOG DATASET SUMMARY")
print("=" * 60)
print(f"Total records        : {len(out_df)}")
print(f"Unique categories    : {out_df['category'].nunique()}")
print()
print("Simulator context richness:")
for score, count in sorted(Counter(out_df['context_richness']).items()):
    labels = {0: 'none (OQ only)', 1: 'FD or agent CQ', 2: 'FD + agent CQ'}
    print(f"  {labels[score]:20s}: {count:3d} ({count/len(out_df):.1%})")
print()
print("Answer sources:")
for src, count in Counter(out_df['answer_source']).items():
    print(f"  {src:25s}: {count:3d} ({count/len(out_df):.1%})")
print()
print("Word counts (mean):")
print(f"  original_question   : {out_df['original_question'].apply(lambda x: len(x.split())).mean():.0f} words")
print(f"  simulator_context   : {out_df['simulator_context'].apply(lambda x: len(x.split()) if x else 0).mean():.0f} words")
print(f"  accepted_answer     : {out_df['accepted_answer'].apply(lambda x: len(x.split())).mean():.0f} words")
print()
print(f"Output file          : {OUTPUT_JSONL}")
print("=" * 60)

MS-DIALOG DATASET SUMMARY
Total records        : 100
Unique categories    : 19

Simulator context richness:
  FD + agent CQ       : 100 (100.0%)

Answer sources:
  fallback_last_pa         :  14 (14.0%)
  explicit_is_answer       :  86 (86.0%)

Word counts (mean):
  original_question   : 84 words
  simulator_context   : 197 words
  accepted_answer     : 81 words

Output file          : D:\final_project\pilot_study\datasets\ms-dialog\msdialog_100.jsonl


## Experiment Design Notes

### How this maps to the Phase 1 pipeline

```
Model sees:
  [title]  — product context (e.g. "Windows 10 — Edge crashes")
  [original_question]  — user's problem description (OQ utterances)
  Prompt: "What is the most helpful solution to this user's problem?"

Model generates:
  preliminary_solution  — initial attempt at resolving the issue
  clarifying_question   — optional CQ to the user
  confidence            — self-reported certainty

Simulator receives:
  simulator_context     — FD utterances + agent CQs (hidden from model)
  clarifying_question   — what the model asked

Evaluator compares:
  final_solution (after CQ rounds) vs accepted_answer
  → binary RESOLVED / NOT RESOLVED verdict
```

### Differences from MedQA
| Aspect | MedQA | MS-Dialog |
|---|---|---|
| Task type | Diagnosis MCQ | Open-ended tech support |
| Ground truth | Correct option (A/B/C/D) | Accepted answer text |
| Evaluation | Exact match | Semantic RESOLVED/NOT RESOLVED |
| Difficulty | Labelled (easy/medium/hard) | Implicit (by category + dialog length) |
| Domain | Clinical medicine | Microsoft tech support |